# Краткое описание
- Цель эксперимента - первичный EDA для корпуса перефразирования с детоксикацией.
- Данные - ru_paradetox (пары токсичных и нейтральных фраз).
- Основные выводы - базовые распределения длины и качества данных, отмечены шумы и дубликаты.


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:


# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train.tsv', 'validation': 'dev.tsv'}
df_train = pd.read_csv("hf://datasets/s-nlp/ru_paradetox/" + splits["train"], sep="\t")
df_val = pd.read_csv("hf://datasets/s-nlp/ru_paradetox/" + splits["validation"], sep="\t")

In [ ]:
df_train.head()

In [ ]:
df_val.head()

In [ ]:
print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)

In [ ]:
df_train.describe(include='object')

In [ ]:
df_val.describe(include='object')

In [ ]:
df_train.info()

In [ ]:
df_val.info()

In [ ]:
print("Train missing values:")
print(df_train.isnull().sum())

print("\nValidation missing values:")
print(df_val.isnull().sum())

In [ ]:
print("Train duplicates:", df_train.duplicated().sum())
print("Validation duplicates:", df_val.duplicated().sum())

In [ ]:
print("Train duplicate rows by toxic comment:", df_train["ru_toxic_comment"].duplicated().sum())
print("Train duplicate rows by neutral comment:", df_train["ru_neutral_comment"].duplicated().sum())

In [ ]:
print("Train duplicate pairs:", df_train.duplicated(subset=["ru_toxic_comment", "ru_neutral_comment"]).sum())
print("Validation duplicate pairs:", df_val.duplicated(subset=["ru_toxic_comment", "ru_neutral_comment"]).sum())

Объединение train и validation для EDA

In [ ]:
df_train["split"] = "train"
df_val["split"] = "validation"

df = pd.concat([df_train, df_val], ignore_index=True)
df.head()

Проверка на аномалии

In [ ]:
text_cols = ["ru_toxic_comment", "ru_neutral_comment"]
len_cols = ["toxic_len_words", "neutral_len_words"]

df_checked = df.copy()



for col in text_cols:
    df_checked[col] = df_checked[col].where(df_checked[col].notna(), "")
    df_checked[col] = df_checked[col].astype(str)

df_checked["toxic_len_words"] = df_checked["ru_toxic_comment"].str.split().str.len()
df_checked["neutral_len_words"] = df_checked["ru_neutral_comment"].str.split().str.len()

for col in len_cols:
    df_checked[col] = pd.to_numeric(df_checked[col], errors="coerce")



anomalies = {}



anomalies["missing_toxic"] = df["ru_toxic_comment"].isna().sum()
anomalies["missing_neutral"] = df["ru_neutral_comment"].isna().sum()

anomalies["empty_toxic"] = (df_checked["ru_toxic_comment"].str.strip() == "").sum()
anomalies["empty_neutral"] = (df_checked["ru_neutral_comment"].str.strip() == "").sum()

anomalies["toxic_len_1"] = (df_checked["toxic_len_words"] == 1).sum()
anomalies["toxic_len_2"] = (df_checked["toxic_len_words"] == 2).sum()
anomalies["neutral_len_1"] = (df_checked["neutral_len_words"] == 1).sum()
anomalies["neutral_len_2"] = (df_checked["neutral_len_words"] == 2).sum()

anomalies["toxic_len_gt_25"] = (df_checked["toxic_len_words"] > 25).sum()
anomalies["neutral_len_gt_25"] = (df_checked["neutral_len_words"] > 25).sum()

anomalies["toxic_len_lt_neutral"] = (df_checked["toxic_len_words"] < df_checked["neutral_len_words"]).sum()
anomalies["toxic_len_eq_neutral"] = (df_checked["toxic_len_words"] == df_checked["neutral_len_words"]).sum()

anomalies["error_like_toxic"] = df_checked["ru_toxic_comment"].str.contains(
    r"error|#ERROR!",
    case=False,
    regex=True,
    na=False
).sum()

anomalies["error_like_neutral"] = df_checked["ru_neutral_comment"].str.contains(
    r"error|#ERROR!",
    case=False,
    regex=True,
    na=False
).sum()

pd.Series(anomalies).sort_values(ascending=False)

In [ ]:
# Подготовка колонок и вычисление token_overlap до проверки аномалий
def tokenize_simple(text):
    if text is None:
        return []
    return str(text).split()

def token_overlap(a, b):
    a_set = set(tokenize_simple(a))
    b_set = set(tokenize_simple(b))
    if len(a_set) == 0 or len(b_set) == 0:
        return 0.0
    return len(a_set & b_set) / len(a_set | b_set)

# Убедимся, что столбцы есть и приведены к строке
for col in ["ru_toxic_comment", "ru_neutral_comment"]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].where(df[col].notna(), "").astype(str)

# Длины в словах (создаём/пересоздаём)
df["toxic_len_words"] = df["ru_toxic_comment"].astype(str).str.split().str.len()
df["neutral_len_words"] = df["ru_neutral_comment"].astype(str).str.split().str.len()

# Вычисляем token_overlap если ещё нет
if "token_overlap" not in df.columns:
    df["token_overlap"] = df.apply(lambda row: token_overlap(row["ru_toxic_comment"], row["ru_neutral_comment"]), axis=1)

# Сформируем маску аномалий и выведем таблицу
anomaly_mask = (
    (df["toxic_len_words"] <= 2)
    | (df["neutral_len_words"] <= 2)
    | (df["toxic_len_words"] > 25)
    | (df["neutral_len_words"] > 25)
    | (df["ru_toxic_comment"].astype(str).str.contains(r"error|ошиб|#ERROR!", case=False, regex=True))
    | (df["ru_neutral_comment"].astype(str).str.contains(r"error|ошиб|#ERROR!", case=False, regex=True))
)

display(
    df.loc[
        anomaly_mask,
        ["ru_toxic_comment", "ru_neutral_comment", "toxic_len_words", "neutral_len_words", "token_overlap"]
    ].sort_values(["toxic_len_words", "neutral_len_words"]).head(20)
)

In [ ]:
error_like_toxic = df[
    df["ru_toxic_comment"]
    .astype(str)
    .str.contains(r"error|#ERROR!", case=False, regex=True, na=False)
][["ru_toxic_comment", "ru_neutral_comment"]]

error_like_neutral = df[
    df["ru_neutral_comment"]
    .astype(str)
    .str.contains(r"error|#ERROR!", case=False, regex=True, na=False)
][["ru_toxic_comment", "ru_neutral_comment"]]

print("error_like_toxic:")
display(error_like_toxic)

print("error_like_neutral:")
display(error_like_neutral)

Длина текстов

In [ ]:
df["toxic_len_chars"] = df["ru_toxic_comment"].str.len()
df["neutral_len_chars"] = df["ru_neutral_comment"].str.len()

df["toxic_len_words"] = df["ru_toxic_comment"].str.split().apply(len)
df["neutral_len_words"] = df["ru_neutral_comment"].str.split().apply(len)

In [ ]:
df[["toxic_len_chars", "neutral_len_chars", "toxic_len_words", "neutral_len_words"]].describe()

Визуализация распределения длины

In [ ]:
from pathlib import Path

def _find_project_dir(name="project", max_levels=10):
    p = Path.cwd()
    for _ in range(max_levels):
        if p.name == name:
            return p
        p = p.parent
    for ancestor in Path.cwd().parents:
        if ancestor.name == name:
            return ancestor
    return Path.cwd()

project_dir = _find_project_dir()
output_dir = project_dir / "artifacts" / "EDA" / "detox_dataset"
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(df["toxic_len_chars"], bins=50, color="red", label="toxic", kde=True, stat="density", alpha=0.4)
sns.histplot(df["neutral_len_chars"], bins=50, color="green", label="neutral", kde=True, stat="density", alpha=0.4)
plt.legend()
plt.title("Distribution of comment length in characters")
plt.xlabel("Characters")
plt.ylabel("Density")
plt.savefig(str(output_dir / "character_length_distribution.png"))
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(df["toxic_len_words"], bins=50, color="red", label="toxic", kde=True, stat="density", alpha=0.4)
sns.histplot(df["neutral_len_words"], bins=50, color="green", label="neutral", kde=True, stat="density", alpha=0.4)
plt.legend()
plt.title("Distribution of comment length in words")
plt.xlabel("Words")
plt.ylabel("Density")
plt.savefig(str(output_dir / "word_length_distribution.png"))
plt.show()

In [ ]:
length_df = pd.DataFrame({
    "type": ["toxic"] * len(df) + ["neutral"] * len(df),
    "length_words": list(df["toxic_len_words"]) + list(df["neutral_len_words"])
})

plt.figure(figsize=(8, 5))
sns.boxplot(data=length_df, x="type", y="length_words")
plt.title("Comment length comparison")
plt.xlabel("")
plt.ylabel("Words")
plt.savefig(str(output_dir / "comment_length_comparison.png"))
plt.show()

Проверка самых коротких и длинных примеров

In [ ]:
df.sort_values("toxic_len_words").head(10)[["ru_toxic_comment", "ru_neutral_comment", "toxic_len_words"]]

In [ ]:
df.sort_values("toxic_len_words", ascending=False).head(10)[["ru_toxic_comment", "ru_neutral_comment", "toxic_len_words"]]

In [ ]:
df.sort_values("neutral_len_words").head(10)[["ru_toxic_comment", "ru_neutral_comment", "neutral_len_words"]]

In [ ]:
df.sort_values("neutral_len_words", ascending=False).head(10)[["ru_toxic_comment", "ru_neutral_comment", "neutral_len_words"]]

Доля изменений между toxic и neutral

In [ ]:
def token_overlap(a, b):
    a_set = set(str(a).lower().split())
    b_set = set(str(b).lower().split())
    if len(a_set) == 0 or len(b_set) == 0:
        return 0
    return len(a_set & b_set) / len(a_set | b_set)

df["token_overlap"] = df.apply(lambda row: token_overlap(row["ru_toxic_comment"], row["ru_neutral_comment"]), axis=1)
df["token_overlap"].describe()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["token_overlap"], bins=50, kde=True)
plt.title("Token overlap between toxic and neutral versions")
plt.xlabel("Jaccard similarity")
plt.ylabel("Count")
plt.savefig(str(output_dir / "token_overlap.png"))
plt.show()

Разница в длине между токсичным и нейтральным текстом

In [ ]:
df["len_diff_words"] = df["toxic_len_words"] - df["neutral_len_words"]
df["len_diff_chars"] = df["toxic_len_chars"] - df["neutral_len_chars"]

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["len_diff_words"], bins=50, kde=True)
plt.title("Difference in word count: toxic - neutral")
plt.xlabel("Words difference")
plt.ylabel("Count")
plt.savefig(str(output_dir / "difference_in_word_count.png"))
plt.show()

Частотный анализ слов

In [ ]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^а-яёa-z0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
df["toxic_clean"] = df["ru_toxic_comment"].apply(preprocess_text)
df["neutral_clean"] = df["ru_neutral_comment"].apply(preprocess_text)

In [ ]:
toxic_words = " ".join(df["toxic_clean"]).split()
neutral_words = " ".join(df["neutral_clean"]).split()

toxic_counter = Counter(toxic_words)
neutral_counter = Counter(neutral_words)

In [ ]:
print("Top toxic words:")
print(toxic_counter.most_common(20))

In [ ]:
print("Top neutral words:")
print(neutral_counter.most_common(20))

График самых частых слов

In [ ]:
top_toxic = pd.DataFrame(toxic_counter.most_common(20), columns=["word", "count"])
top_neutral = pd.DataFrame(neutral_counter.most_common(20), columns=["word", "count"])

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_toxic, y="word", x="count", color="red")
plt.title("Top 20 toxic words")
plt.xlabel("Count")
plt.ylabel("")
plt.savefig(str(output_dir / "top_20_words_in_toxic.png"))
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_neutral, y="word", x="count", color="green")
plt.title("Top 20 neutral words")
plt.xlabel("Count")
plt.ylabel("")
plt.savefig(str(output_dir / "top_20_words_in_neutral.png"))
plt.show()

Самые частые биграммы

In [ ]:
def get_top_ngrams(text_series, n=2, top_k=20):
    vectorizer = CountVectorizer(ngram_range=(n, n))
    X = vectorizer.fit_transform(text_series)
    freqs = X.sum(axis=0).A1
    ngrams = vectorizer.get_feature_names_out()
    result = pd.DataFrame({"ngram": ngrams, "count": freqs})
    return result.sort_values("count", ascending=False).head(top_k)

In [ ]:
top_toxic_bigrams = get_top_ngrams(df["toxic_clean"], n=2, top_k=20)
top_toxic_bigrams

In [ ]:
top_neutral_bigrams = get_top_ngrams(df["neutral_clean"], n=2, top_k=20)
top_neutral_bigrams

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_toxic_bigrams, y="ngram", x="count", color="red")
plt.title("Top 20 toxic bigrams")
plt.xlabel("Count")
plt.ylabel("")
plt.savefig(str(output_dir / "top_20_toxic_bigrams.png"))
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_neutral_bigrams, y="ngram", x="count", color="green")
plt.title("Top 20 neutral bigrams")
plt.xlabel("Count")
plt.ylabel("")
plt.savefig(str(output_dir / "top_20_neutral_bigrams.png"))
plt.show()
